# Build a Wordle Solver
### UC Berkeley EECS Day · CS Lab

Welcome! In the next hour you'll **teach a computer to play Wordle**. Then you'll try to make your solver strong enough to **beat a Berkeley student's solver** in our mini competition.

**How this notebook works**
* To **run a cell**, click on it and press **Shift + Enter** (or click the play button on its left).
* Run the cells **in order, from top to bottom**. Later cells use code from earlier ones.
* Most of the code is already written. Wherever you see **`___`**, replace it with the missing code.
* Most exercises end with a **check** that tells you whether you got it right.
* If you see `NameError: name '___' is not defined`, there's still a blank you need to fill in.
* Stuck? Raise your hand. Helping you is what the TAs are here for!

## Part 0: Setup

Run the cell below. It runs the setup file `wordle_setup.py`, which creates our dataset, `wordbank.txt` (all **2,315** official Wordle answers), and loads some helper tools for later.

If it asks you to upload `wordle_setup.py`, click **Choose Files** and pick the file your TA gave you.

In [ ]:
# @title Run me first! (loads the word bank + helper tools)
import os, urllib.request

SETUP_URL = "https://raw.githubusercontent.com/alexmenchtchikov/EECS-Day-CS-Lab/main/wordle_setup.py"

if not os.path.exists("wordle_setup.py") and SETUP_URL:
    urllib.request.urlretrieve(SETUP_URL, "wordle_setup.py")
if not os.path.exists("wordle_setup.py"):
    from google.colab import files
    print("Please upload wordle_setup.py (ask a TA if you can't find it).")
    files.upload()

%run -i wordle_setup.py

## Part 1: Python warm-up

Let's review the Python we'll need. Every one of these ideas shows up in the solver!

### 1.1 Python basics

**Variables** store values so you can use them later. `print` shows them on the screen.

In [ ]:
name = "Oski"           # a string (text) goes in quotes
guesses = 4             # a number doesn't
print(name, "solved it in", guesses, "guesses")

**An `if` statement** runs code only when a condition is `True`. Use `elif` ("else if") to check another condition and `else` for everything else.

* The **indented** lines belong to the `if`. Python uses indentation, not brackets, to group code.
* `==` checks whether two things are equal. A single `=` stores a value in a variable.
* Other comparisons: `!=` (not equal), `<`, `>`, `<=`, `>=`

Try changing `guesses` and running the cell again.

In [ ]:
guesses = 4

if guesses == 1:
    print("Genius!")
elif guesses <= 4:
    print("Nice job!")
else:
    print("Phew, you got it!")

**A `for` loop** repeats code once for each item in a list (or each letter in a string). `range(n)` gives the numbers `0, 1, ..., n-1`.

In [ ]:
for word in ["crane", "slate", "pinky"]:
    print("Guessing", word)

for i in range(3):      # i is 0, then 1, then 2
    print("Turn number", i)

**A `while` loop** keeps repeating as long as its condition is `True`. We'll use one to keep guessing until the game is won.

In [ ]:
guesses_left = 3
while guesses_left > 0:
    print(guesses_left, "guesses left")
    guesses_left = guesses_left - 1
print("Out of guesses!")

**A list** holds many values in order. **A function** gives a piece of code a name. It takes inputs (called *parameters*) and can `return` an answer.

In [ ]:
words = ["crane", "slate"]
words.append("pinky")           # add to the end of the list
print(words, "has", len(words), "words. The first is", words[0])

def double(x):                  # x is the input
    return x * 2                # send the answer back

print(double(21))

### 1.2 Loading the dataset (string cleanup)

Real data is usually messy. Let's look at the first 30 characters of `wordbank.txt`:

In [ ]:
raw_text = open("wordbank.txt").read()
print(repr(raw_text[:30]))

Each word is on its own line, so each line ends with `\n` (the "newline" character). The words are also in UPPERCASE. We want clean lowercase words like `"aback"`.

Two **string methods** will help:

In [ ]:
messy = "  CRANE\n"
print(repr(messy))
print(repr(messy.strip()))            # .strip() removes spaces and newlines from both ends
print(repr(messy.lower()))            # .lower() makes every letter lowercase
print(repr(messy.strip().lower()))    # you can chain methods together!

**Exercise:** Fill in the blank so every line becomes a clean, lowercase word.

In [ ]:
WORDS = []
with open("wordbank.txt") as file:
    for line in file:              # each line looks like "ABACK\n"
        word = ___        # clean up the line
        WORDS.append(word)

print("First 5 words:", WORDS[:5])
check_wordbank(WORDS)

### 1.3 Strings are sequences of letters

A string works a lot like a list of letters. You can grab a letter by its **index** (counting starts at **0**!), loop over the letters, and check whether a letter is `in` it.

In [ ]:
word = "crane"
print(word[0])        # first letter: c
print(word[4])        # fifth letter: e
print(len(word))      # number of letters: 5
print("r" in word)    # True
print("z" in word)    # False

for letter in word:   # loop over each letter
    print(letter, end=" ")

**Exercise:** Finish `count_vowels` so it counts how many letters in `word` are vowels.
*Hint:* The string `"aeiou"` holds every vowel. How can you check whether `letter` is one of them?

In [ ]:
def count_vowels(word):
    count = 0
    for letter in word:
        if ___:
            count = count + 1
    return count

if check_count_vowels(count_vowels):
    print('count_vowels("audio") =', count_vowels("audio"))

### 1.4 Comparing two words letter by letter

Sometimes we need the **position** of each letter, not just the letter. `range(5)` gives us the positions `0, 1, 2, 3, 4`.

**Exercise:** Count how many positions have the **same letter** in both words. For example, `crane` and `trace` match in 3 spots: \_ **R A** \_ **E**. That's exactly how Wordle finds green letters!

In [ ]:
def count_same_position(word1, word2):
    count = 0
    for i in range(5):
        if ___:
            count += 1             # += 1 is a shortcut for count = count + 1
    return count

if check_count_same_position(count_same_position):
    print('count_same_position("crane", "trace") =', count_same_position("crane", "trace"))

### 1.5 Filtering a list

Say Wordle tells you there's **no E** in the secret word. Which words are still possible? We loop over the list and keep only the words we want.

**Exercise:** Keep only the words that do **not** contain `letter`. *Hint:* Python has a `not in` operator.

In [ ]:
def words_without(words, letter):
    result = []
    for word in words:
        if ___:
            result.append(word)    # .append adds an item to the end of a list
    return result

if check_words_without(words_without):
    print("Words with no 'e':", len(words_without(WORDS, "e")), "out of", len(WORDS))

Filtering a list like this is the **heart** of our Wordle solver. Each guess filters out words that can't be the answer.

### 1.6 Dictionaries: counting things

A **dictionary** stores `key: value` pairs. It's perfect for counting, like *"how many times does each letter appear?"*

**Exercise:** Finish `letter_counts`. If we've seen a letter before, add 1 to its count. If not, its count starts at 1.

In [ ]:
def letter_counts(word):
    counts = {}                       # an empty dictionary
    for letter in word:
        if letter in counts:          # seen this letter before?
            counts[letter] = ___
        else:                         # first time seeing it
            counts[letter] = ___
    return counts

if check_letter_counts(letter_counts):
    print('letter_counts("speed") =', letter_counts("speed"))

Two handy shortcuts we'll use later:

In [ ]:
counts = {"e": 2}
print(counts.get("e", 0))   # .get(key, default) returns the value if the key exists...
print(counts.get("z", 0))   # ...and returns the default (0 here) if it doesn't. No error!

print(set("speed"))         # set() keeps only the UNIQUE letters (in no particular order)

## Part 2: Play Wordle!

**Rules:** Guess the secret 5-letter word in **6 tries**. After each guess, every letter gets a color:

* 🟩 **Green:** right letter, right spot
* 🟨 **Yellow:** the letter is in the word, but in a different spot
* ⬛ **Gray:** the letter isn't in the word (or every copy of it is already accounted for)

Our version picks a random secret from the **same word bank** you just loaded, and every guess must be a word in the bank. Click the game, then type. Press **New game** to play again.

**While you play, think about how *you* decide what to guess next.** That's the algorithm we're going to teach the computer!

In [ ]:
play_wordle()

In code, we'll write the colors as a **5-character string**: `G` = green, `Y` = yellow, `-` = gray.
For example, guessing **CRANE** when the secret is **TRACE** gives `"YGG-G"`:

In [ ]:
print(emoji("YGG-G"))
show_board([("crane", "YGG-G")])

## Part 3: Teach the computer Wordle's colors

Let's write `get_feedback(guess, secret)`, which returns the color string for a guess.

**The tricky part is repeated letters.** Suppose the secret is **ABIDE** and you guess **SPEED**. SPEED has *two* E's, but ABIDE only has *one*, so only **one** E can light up:

In [ ]:
show_board([("speed", "--Y-Y")], title="secret: ABIDE")

To handle this, real Wordle uses **two passes**:

1. **Greens first.** For each position, if the letters match, it's `G`. Every secret letter that *isn't* matched goes into a `leftover` dictionary, which counts how many copies of each letter are still unclaimed.
2. **Then yellows.** For each position that isn't green, if the guessed letter is still in `leftover` (count > 0), it's `Y`, and we **use up** one copy so it can't be counted twice.

Anything left over stays gray: `-`.

**Exercise:** Fill in the three blanks.

In [ ]:
def get_feedback(guess, secret):
    feedback = ["-", "-", "-", "-", "-"]   # start with all gray
    leftover = {}                          # secret letters that are NOT green

    # Pass 1: greens
    for i in range(5):
        if ___:
            feedback[i] = "G"
        else:
            leftover[secret[i]] = leftover.get(secret[i], 0) + 1

    # Pass 2: yellows
    for i in range(5):
        if feedback[i] == "G":
            continue                       # already green, so skip to the next position
        letter = guess[i]
        if leftover.get(letter, 0) > 0:    # an unclaimed copy of this letter exists!
            feedback[i] = ___
            leftover[letter] = ___   # use up that copy

    return "".join(feedback)               # turn the list into a string, like "YGG-G"

check_feedback(get_feedback)

Try it yourself! Change the guess and the secret below.

In [ ]:
guess, secret = "speed", "abide"
fb = get_feedback(guess, secret)
print(fb, emoji(fb))
show_board([(guess, fb)], title=f"secret: {secret.upper()}")

## Part 4: Narrow down the possibilities

Here's the **key idea** behind every Wordle solver:

> After a guess, go through every word and ask: ***"If this word were the secret, would my guess have gotten the exact same colors?"*** If not, it **can't** be the secret, so cross it off!

The nice part is that we don't need separate rules for greens, yellows, and grays, because `get_feedback` already knows them all.

**Exercise:** Keep only the words that are still possible.

In [ ]:
def filter_words(candidates, guess, feedback):
    remaining = []
    for word in candidates:
        # If `word` were the secret, would `guess` get the same feedback?
        if ___:
            remaining.append(word)
    return remaining

check_filter_words(filter_words)

Watch the possibilities shrink! The secret here is **PINKY**:

In [ ]:
secret = "pinky"
remaining = list(WORDS)
print(f"Start: {len(remaining)} possible words")
for guess in ["raise", "count"]:
    fb = get_feedback(guess, secret)
    remaining = filter_words(remaining, guess, fb)
    print(f"Guess {guess.upper()} {emoji(fb)}  →  {len(remaining)} words left")
print(remaining)

**Wordle helper:** Use this on a real Wordle game! Type in your guesses and the colors you got (`G`, `Y`, `-`), and it lists the words that are still possible.

In [ ]:
my_game = [
    ("raise", "--Y--"),     # (guess, feedback) pairs, add as many as you like
    ("count", "---Y-"),
]

remaining = list(WORDS)
for guess, fb in my_game:
    remaining = filter_words(remaining, guess, fb)
print(len(remaining), "possible words:", remaining[:50])

## Part 5: Your first solver

A **solver** is a function that decides what to guess next. Every solver in this notebook looks like this:

```python
def some_solver(candidates, history):
    ...
    return guess
```

* `candidates` is the list of words that are **still possible** (they all fit the colors so far).
* `history` is a list of `(guess, feedback)` pairs from earlier in the game, like `[("raise", "--Y--")]`. It's empty on the first turn.
* It returns a 5-letter word from the word bank.

### 5.1 The simplest possible strategy

Every word in `candidates` could be the answer, so just guess the **first** one!

**Exercise:**

In [ ]:
def first_word_solver(candidates, history):
    return ___

check_first_word_solver(first_word_solver)

### 5.2 The game loop

Now let's make the computer play a whole game. Each turn:
1. ask the solver for a guess,
2. get the feedback,
3. stop if every letter is green (`"GGGGG"`),
4. otherwise, filter the candidates and go again.

**Exercise:** Fill in the three blanks using functions you've already written.

In [ ]:
def play_game(solver, secret, show=True):
    candidates = list(WORDS)      # at the start, every word is possible
    history = []                  # the (guess, feedback) pairs so far

    while len(history) < 20:      # safety limit so we never loop forever
        guess = solver(candidates, history)
        feedback = ___
        history.append((guess, feedback))
        if show:
            print(f"Guess {len(history)}: {guess.upper()} {emoji(feedback)}   (possible words: {len(candidates)})")

        if ___:
            break                 # solved it!
        candidates = ___

    if show:
        show_board(history, title=f"Solved in {len(history)} guesses!")
    return len(history)

check_play_game(play_game)

Watch your solver play a random word. Run this cell a few times!

In [ ]:
secret = random.choice(WORDS)
print("The secret word is:", secret.upper(), "\n")
play_game(first_word_solver, secret)

### 5.3 How good is it?

One game doesn't tell us much. Let's play **100 games** and find the **average number of guesses** (lower is better). `test_solver` always uses the same 100 secret words, so the comparison between solvers is fair.

In [ ]:
test_solver(first_word_solver)

Not bad! But the first guess is always **ABACK** (it's first alphabetically). That word uses rare letters (B, K) and repeats the A. We can do better...

## Part 6: A smarter solver: common letters

**Idea:** Guess words made of the **most common letters** among the remaining candidates. If you test common letters, every answer (green, yellow, *or* gray) rules out lots of words.

**The plan:**
1. For each letter, count how many candidates contain it.
2. Give each candidate a **score**: add up the counts of its *unique* letters (`set(word)`, so repeated letters don't count twice).
3. Guess the candidate with the **highest score**.

For example, if 1,000 candidates contain an E and only 30 contain a Z, an E is worth 1,000 points and a Z only 30.

**Exercise:** Fill in the three blanks.

In [ ]:
def letter_frequency_solver(candidates, history):
    # Step 1: count how many candidates contain each letter
    counts = {}
    for word in candidates:
        for letter in set(word):
            counts[letter] = ___

    # Step 2 + 3: score every candidate and remember the best one
    best_word = None
    best_score = -1
    for word in candidates:
        score = 0
        for letter in set(word):
            score = score + ___
        if ___:            # found a new best word?
            best_word = word
            best_score = score
    return best_word

if check_frequency_solver(letter_frequency_solver):
    print("Its first guess is always:", letter_frequency_solver(WORDS, []).upper())

In [ ]:
play_game(letter_frequency_solver, random.choice(WORDS))

In [ ]:
test_solver(letter_frequency_solver)

**You built a real Wordle solver!** Compare its average to `first_word_solver`. Every bit lower is a big deal when you play thousands of games.

The Berkeley student's solver is a little better than this one, though. You have two options:
* **Bonus:** level up your solver with the ideas below, or
* **Competition:** jump to the very bottom and see how `letter_frequency_solver` does against the Berkeley student.

---
## Bonus: Level up your solver

Ideas are listed from easiest to hardest. Try any of them, mix them together, or invent your own! Use `test_solver(...)` to check whether a change actually helps. **Lower average = better.**

### Idea 1: Measure how good a guess is

A great guess **splits the candidates into lots of small groups**. Whatever colors come back, only a few words will be left.

Imagine we guess `guess`. Each candidate secret would give some feedback pattern, so we group the candidates by pattern. If the real secret is in a group of size `k`, then `k` words will be left. That happens with probability `k / n` (where `n = len(candidates)`). So:

$$\text{average words left} = \sum_{\text{groups}} \frac{k}{n} \cdot k = \frac{\sum k^2}{n}$$

**Exercise:** Fill in the blank.

In [ ]:
def average_remaining(guess, candidates):
    """If we guess `guess`, about how many candidates will be left afterward?"""
    groups = {}                          # feedback pattern -> how many candidates give it
    for secret in candidates:
        pattern = get_feedback(guess, secret)
        groups[pattern] = groups.get(pattern, 0) + 1

    total = 0
    for k in groups.values():
        total = total + ___
    return total / len(candidates)

check_average_remaining(average_remaining)

### Idea 2: Pick a great first guess

On the first turn every word is still possible, so the best first guess is the **same in every game**. Find a great one once, then use it every time! Which of these openers is best? (Lower = better.) Add your own ideas to the list.

In [ ]:
for opener in ["audio", "house", "ghost", "crane", "pizza"]:
    print(opener, "→", round(average_remaining(opener, WORDS), 1), "words left on average")

**Challenge:** Search *all 2,315 words* for the best opener. (This checks 2,315 × 2,315 ≈ 5 million feedbacks, so it can take up to a minute.)

Watch out: the opener that looks best here doesn't always give the best `test_solver` average, because that also depends on what your solver does *after* the first guess. Always test it!

In [ ]:
# best = min(WORDS, key=lambda w: average_remaining(w, WORDS))   # un-comment (remove the #) to run
# print(best, average_remaining(best, WORDS))

In [ ]:
MY_OPENER = "crane"    # change this to the best opener you found!
if MY_OPENER not in WORDS:
    print(f"Warning: {MY_OPENER!r} isn't in the word bank, so it can't be guessed. Pick another word!")

def opener_solver(candidates, history):
    if len(history) == 0:              # first turn?
        return MY_OPENER
    return letter_frequency_solver(candidates, history)

test_solver(opener_solver)

### Idea 3: Pick the guess that leaves the fewest words

Instead of counting letters, score every candidate with `average_remaining` and guess the one with the **smallest** score.

In [ ]:
def smart_solver(candidates, history):
    if len(history) == 0:
        return MY_OPENER
    best_guess = None
    best_score = float("inf")          # infinity: any real score will be smaller
    for guess in candidates:
        score = ___
        if score < best_score:
            best_guess = guess
            best_score = score
    return best_guess

test_solver(smart_solver)

### Idea 4: Sometimes guess a word that *can't* be the answer

Suppose the candidates are **BATCH, HATCH, LATCH, MATCH, PATCH, WATCH**. If you only guess candidates, you might need 6 guesses! A word like **PLUMB** isn't a candidate, but it tests P, L, M, and B all at once. Whatever colors come back, at most two words are left.

**Try it:** In `smart_solver`, loop over **all** of `WORDS` instead of just `candidates`. Some tips:
* Give real candidates a small bonus (e.g. `score -= 0.1`), because they might win right away.
* When only 1 or 2 candidates are left, just guess one of them!
* This is slower, so test with fewer games: `test_solver(my_solver, num_games=30)`.

### Idea 5: Ideas for the ambitious
* **Position matters:** in the letter-frequency solver, give extra points when a letter is common *in that exact spot* (lots of words end in E, S, or Y!).
* **Information theory:** score guesses by *entropy*, the expected number of "bits" of information you gain. Look up 3Blue1Brown's video *"Solving Wordle using information theory"* on YouTube.
* **Look ahead:** for the top few guesses, simulate the *next* turn too, and pick the guess with the best two-turn result.
* **Combine ideas:** use a smart, slow strategy when few candidates are left, and a fast strategy when there are many.

### Your solver
Build your best solver here (or make `my_solver` just call one of the solvers above).

In [ ]:
def my_solver(candidates, history):
    # Make this your best solver! Right now it just copies the letter-frequency solver.
    if len(history) == 0:
        return MY_OPENER
    return letter_frequency_solver(candidates, history)

test_solver(my_solver)

---
## Competition: Can you outperform a Berkeley student?

* Your solver plays **20 secret words** from the word bank. The **Berkeley student's solver** plays the exact same 20 words.
* Your score is your **average number of guesses**. **Lower wins!**
* Solvers get the usual `candidates` and `history`. A game that isn't solved within 10 guesses counts as 12.
* Your TA will announce the **competition code** so everyone plays the same secret words. Type it in below.
* No hard-coding the secret words!

Change `SOLVER` to your best solver (e.g. `my_solver`, `smart_solver`, `letter_frequency_solver`), then run the cell.

In [ ]:
COMPETITION_CODE = 90               # practice code. Your TA will announce the real one!
SOLVER = letter_frequency_solver    # change this to your best solver

competition(SOLVER, COMPETITION_CODE)